In [1]:
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [ ]:
# run_raster_parallel.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from raster_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()

    #sessions=["ecephys_776293_2025-02-19_14-01-07_sorted_2025-03-30_08-55-28"]
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


In [ ]:
# move files to different subfolders
from pathlib import Path
import re
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = Path("/root/capsule/scratch/raster_plot")
DRY_RUN = False

PNG_RE = re.compile(r"^(?P<prefix>.+)_unit_(?P<unit>\d+)\.png$", re.IGNORECASE)

def move_one(png_path: Path, dest_path: Path) -> None:
    if DRY_RUN:
        return
    png_path.rename(dest_path)

moved = 0
skipped = 0

max_workers = min(32, (os.cpu_count() or 8) * 2)
PRINT_EVERY = 500  # adjust

for session_dir in (p for p in ROOT.iterdir() if p.is_dir()):
    created = set()
    tasks = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for png_path in session_dir.glob("*.png"):
            m = PNG_RE.match(png_path.name)
            if not m:
                skipped += 1
                continue

            prefix = m.group("prefix")
            dest_dir = session_dir / prefix

            if prefix not in created:
                if not DRY_RUN:
                    dest_dir.mkdir(parents=True, exist_ok=True)
                created.add(prefix)

            dest_path = dest_dir / png_path.name
            tasks.append(ex.submit(move_one, png_path, dest_path))

        total = len(tasks)
        done = 0

        for fut in as_completed(tasks):
            fut.result()
            moved += 1
            done += 1

            if done % PRINT_EVERY == 0 or done == total:
                print(f"[{session_dir.name}] {done}/{total} done | moved_total={moved} skipped={skipped}")

print("\nSummary")
print("Moved:", moved)
print("Skipped:", skipped)


In [ ]:
# scatter_activity_latent_worker.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from scatter_activity_latent_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


In [2]:
# Run raster plots for a specific list of sessions
import os
import multiprocessing as mp
import concurrent.futures as cf
from raster_worker import process_session, OUTDIR

sessions = [
    "ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14",
    "ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17",
    "ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58",
    "ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38",
    "ecephys_839483_2026-05-26_16-02-39_sorted_2026-06-09_09-42-36",
    "ecephys_839483_2026-05-28_14-47-06_sorted-bandpass_2026-07-17_20-18-46",
]

OUTDIR.mkdir(parents=True, exist_ok=True)
print("Sessions:", sessions)
ctx = mp.get_context("spawn")
max_workers = min(len(sessions), os.cpu_count() or 1)
with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
    futures = {ex.submit(process_session, s): s for s in sessions}
    for fut in cf.as_completed(futures):
        print(fut.result())

Sessions: ['ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14', 'ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17', 'ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58', 'ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38', 'ecephys_839483_2026-05-26_16-02-39_sorted_2026-06-09_09-42-36', 'ecephys_839483_2026-05-28_14-47-06_sorted-bandpass_2026-07-17_20-18-46']
No model-fitting results found for subject 839480 on 2026-06-04

[ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58] models: None
[ecephys_839483_2026-05-26_16-02-39_sorted_2026-06-09_09-42-36] skip: Zarr not found: /root/capsule/scratch/psth_results/ecephys_839483_2026-05-26_16-02-39_sorted_2026-06-09_09-42-36_0.2s.zarr
No model-fitting results found for subject 839480 on 2026-06-05

[ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38] models: None
No model-fitting results found for subject 839480 on 2026-06-02

[ec